# 00 — Coordinate Ranges

**Latitude** and **longitude** are the two numbers that describe every point on Earth.

Before you compute distances, draw bounding boxes, or do anything spatial, you need to know what these numbers mean, what valid values look like, and — critically — which order they come in.

Bad coordinates are sneaky. They often look fine. They are just wrong enough to waste your afternoon.

## 1. What Are Latitude and Longitude?

**Latitude** measures north–south position. It ranges from **-90 to 90**.

- `0` is the equator
- Positive values are north of the equator
- Negative values are south

**Longitude** measures east–west position. It ranges from **-180 to 180**.

- `0` is the prime meridian (runs through Greenwich, England)
- Positive values are east of the prime meridian
- Negative values are west — which includes the entire continental United States

```text
Latitude  : -90  to  90   (south to north)
Longitude : -180 to 180   (west to east)
```

A useful memory anchor: latitude lines run **horizontally** (like rungs on a ladder), but they measure **vertical** position. Longitude lines run **vertically**, but they measure **horizontal** position. Geography is committed to being annoying.

## 2. Coordinate Order in GeoJSON

This is the one that trips everyone up at least once.

In GeoJSON, coordinates are stored as `[longitude, latitude]` — **not** `[latitude, longitude]`.

```python
[-98.5, 33.8]   # [longitude, latitude]
```

This is `longitude` first because GeoJSON follows the mathematical `[x, y]` convention — and longitude maps to the horizontal (x) axis. Unfortunately, most human intuition, and many other tools, uses `[lat, lon]` order.

| Format      | Order             | Example          |
|-------------|-------------------|------------------|
| GeoJSON     | `[lon, lat]`      | `[-98.5, 33.8]`  |
| Most humans | `[lat, lon]`      | `[33.8, -98.5]`  |
| Google Maps | `lat, lon`        | `33.8, -98.5`    |

**The rule for this course:** when working with GeoJSON, always read coordinates as `[lon, lat]`.

In [ ]:
# A GeoJSON coordinate pair: [longitude, latitude]
coord = [-98.5, 33.8]

lon = coord[0]
lat = coord[1]

print(f"Longitude: {lon}")   # west of prime meridian → negative
print(f"Latitude:  {lat}")   # north of equator → positive

## 3. Valid Coordinate Ranges

A coordinate is only valid if it falls within the legal ranges for its axis.

| Axis      | Min   | Max  |
|-----------|-------|------|
| Latitude  | -90   | 90   |
| Longitude | -180  | 180  |

Values outside these ranges don't correspond to any location on Earth. If you see `lat = 91` or `lon = -190`, something went wrong — a unit conversion, a swapped pair, a copy-paste error, or a data pipeline that silently corrupted values.

A negative longitude is not a bad sign — it just means west of the prime meridian. Most of North America has negative longitudes. Don't flag those as errors.

In [ ]:
def is_valid_lat(lat):
    return -90 <= lat <= 90

def is_valid_lon(lon):
    return -180 <= lon <= 180

def is_valid_coord(lon, lat):
    return is_valid_lon(lon) and is_valid_lat(lat)


# Test cases
candidates = [
    (33.87,  -98.52),   # valid — north Texas
    (45.0,  -190.0),    # invalid longitude
    (91.0,   22.0),     # invalid latitude
    (-98.5,  33.8),     # looks like swapped [lat, lon] — but both happen to be valid values
]

for lon, lat in candidates:
    valid = is_valid_coord(lon, lat)
    print(f"lon={lon:7.2f}, lat={lat:7.2f}  →  {'valid' if valid else 'INVALID'}")

## 4. Reading Coordinates from GeoJSON

GeoJSON geometries store coordinates differently depending on geometry type. You need to know how to reach them.

**Point** — a single `[lon, lat]` pair:
```json
{ "type": "Point", "coordinates": [-98.5, 33.8] }
```

**LineString** — a list of `[lon, lat]` pairs:
```json
{ "type": "LineString", "coordinates": [[-98.5, 33.8], [-97.2, 34.1]] }
```

**Polygon** — a list of rings, each a list of `[lon, lat]` pairs. The first ring is the exterior:
```json
{ "type": "Polygon", "coordinates": [[[-98.5, 33.8], [-97.2, 33.8], [-97.2, 34.1], [-98.5, 33.8]]] }
```

The nesting depth increases with each geometry type. Points are flat. Polygons have an extra list wrapping the rings.

In [ ]:
import json

feature_collection = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [-98.5, 33.8]},
            "properties": {"name": "Wichita Falls"}
        },
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [-99.0, 33.5],
                    [-97.5, 33.5],
                    [-97.5, 34.5],
                    [-99.0, 34.5],
                    [-99.0, 33.5]
                ]]
            },
            "properties": {"name": "Rough Bounding Box"}
        }
    ]
}

for feature in feature_collection["features"]:
    geom = feature["geometry"]
    name = feature["properties"]["name"]
    gtype = geom["type"]

    if gtype == "Point":
        lon, lat = geom["coordinates"]
        print(f"{name} (Point): lon={lon}, lat={lat}")

    elif gtype == "Polygon":
        exterior = geom["coordinates"][0]   # first ring = exterior
        print(f"{name} (Polygon): {len(exterior)} vertices")
        for coord in exterior:
            print(f"  lon={coord[0]}, lat={coord[1]}")

## 5. Finding Min/Max by Inspection

Before computing a formal bounding box, it is useful to ask four simple questions about a set of coordinates:

- What is the **smallest longitude**? (westernmost point)
- What is the **largest longitude**? (easternmost point)
- What is the **smallest latitude**? (southernmost point)
- What is the **largest latitude**? (northernmost point)

These four values are the building blocks of a bounding box. This section builds that intuition manually before we automate it.

In [ ]:
# [longitude, latitude] — GeoJSON order
points = [
    [-98.5,  33.8],
    [-97.2,  34.1],
    [-99.0,  32.9],
    [-98.1,  33.4],
    [-97.8,  34.6],
]

lons = [p[0] for p in points]
lats = [p[1] for p in points]

print(f"Westernmost (min lon): {min(lons)}")
print(f"Easternmost (max lon): {max(lons)}")
print(f"Southernmost (min lat): {min(lats)}")
print(f"Northernmost (max lat): {max(lats)}")

## 6. Basic Range Checking

Range checking is how you catch problems before they corrupt downstream results. The goal is simple: scan a list of coordinates and report anything that falls outside valid world bounds.

Two common failure modes to check for:
- **Out-of-range values** — longitude beyond ±180 or latitude beyond ±90
- **Suspiciously swapped pairs** — a value that is valid on its own axis but looks wrong for the expected region (e.g., `lon=33.8, lat=-98.5` for Texas data)

In [ ]:
def check_coordinates(coords):
    """
    coords: list of [lon, lat] pairs
    Prints any pairs that fail basic validity checks.
    """
    for i, pair in enumerate(coords):
        lon, lat = pair
        issues = []

        if not (-180 <= lon <= 180):
            issues.append(f"lon {lon} out of range")
        if not (-90 <= lat <= 90):
            issues.append(f"lat {lat} out of range")

        # Heuristic: if lat looks like a longitude and vice versa, flag it
        # (useful when working with US data where lon ~ -60 to -130, lat ~ 20 to 50)
        if -90 <= lon <= 90 and (lat < -90 or lat > 90):
            issues.append("possible swapped [lat, lon]")

        if issues:
            print(f"[{i}] {pair}  →  {'; '.join(issues)}")
        else:
            print(f"[{i}] {pair}  →  ok")


suspect_data = [
    [-98.5,  33.8],   # valid
    [-190.0, 45.0],   # invalid longitude
    [33.8,  -98.5],   # plausible swap
    [91.0,   22.0],   # invalid latitude
    [-97.2,  34.1],   # valid
]

check_coordinates(suspect_data)

## 7. Common Mistakes

These are the things that happen to everyone. Knowing about them in advance does not prevent them. It just reduces the time spent staring at a map wondering why your points are in the Indian Ocean.

**Swapping lat and lon**
The single most common mistake. GeoJSON says `[lon, lat]`. Every instinct says `[lat, lon]`. Your points will be wrong, but valid. They will appear on the map, just in the ocean somewhere near Madagascar.

**Assuming `(x, y)` means `(lat, lon)`**
In math, `(x, y)` is `(horizontal, vertical)`. That maps to `(lon, lat)` — not `(lat, lon)`. If you see a tuple labeled `(x, y)`, it's `(lon, lat)`.

**Treating all geometry types the same**
A `Point` has one coordinate pair. A `LineString` has a list. A `Polygon` has a list of lists (one per ring). Indexing them the same way will either crash or silently return the wrong thing.

**Assuming a flat coordinate list**
Not all coordinate lists are `[[lon, lat], [lon, lat], ...]`. Polygons are `[[[lon, lat], ...]]`. Nested one level deeper. This is the source of a lot of `TypeError: cannot unpack non-sequence float` errors.

**Thinking negative longitude means bad data**
It doesn't. The entire western hemisphere has negative longitudes. `-98.5` is Wichita Falls, not an error.

---

## Exercise A — Label Each Coordinate Pair

For each pair below, determine whether it is **valid**, **invalid** (out of range), or **suspiciously swapped**.

```python
pairs = [
    [-87.6,  41.8],   # Chicago
    [41.8,  -87.6],   # ?
    [180.1,  22.0],   # ?
    [-73.9,  40.7],   # New York
    [-90.0, -91.0],   # ?
    [0.0,    0.0],    # ?
]
```

Write a loop that prints a label for each one.

In [ ]:
pairs = [
    [-87.6,  41.8],
    [41.8,  -87.6],
    [180.1,  22.0],
    [-73.9,  40.7],
    [-90.0, -91.0],
    [0.0,    0.0],
]

# your code here

## Exercise B — Compass Extremes from a Feature Collection

Given the feature collection below, print the four compass extremes: westernmost, easternmost, southernmost, northernmost longitude/latitude values.

In [ ]:
fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.2, 34.1]}, "properties": {}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-99.0, 32.9]}, "properties": {}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.1, 33.4]}, "properties": {}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.8, 34.6]}, "properties": {}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.5, 33.8]}, "properties": {}},
    ]
}

# your code here
# expected output:
#   Westernmost: -99.0
#   Easternmost: -97.2
#   Southernmost: 32.9
#   Northernmost: 34.6

## Exercise C — Scan and Report Invalid Coordinates

The list below contains some coordinate pairs with errors. Write a function `scan_for_errors(coords)` that returns only the pairs that fail validity checks, along with a description of what is wrong.

In [ ]:
mixed_bag = [
    [-98.5,  33.8],
    [-200.0, 45.0],
    [0.0,    91.5],
    [-73.9,  40.7],
    [33.8,  -98.5],
    [-97.0,  35.2],
    [181.0,  22.0],
]

def scan_for_errors(coords):
    # your code here
    pass

errors = scan_for_errors(mixed_bag)
for e in errors:
    print(e)

---

## Check Your Understanding

A dataset claims to contain GeoJSON points for locations in the United States. You pull the first coordinate from one feature:

```python
coord = feature["geometry"]["coordinates"]
print(coord)   # [33.4, -97.1]
```

**Question:** Should you be suspicious of this coordinate pair? Why or why not? What would you check first?

```python
# your answer here
```


---

## Next

In [01 — Compute a Bounding Box](./01-Compute_BBox.ipynb), we formalize the four compass extremes you found above into a reusable `compute_bbox` function and apply it to real GeoJSON data.